In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

# ==========================================
# [0. 필수 함수 정의] - 파일 로드 및 기초 처리
# ==========================================
def load_cmapss_data(data_id, data_path='../CMAPSSData/'):
    cols = ['unit_nr', 'time_cycles', 'setting_1', 'setting_2', 'setting_3'] + [f's_{i}' for i in range(1, 22)]
    train = pd.read_csv(f'{data_path}train_{data_id}.txt', sep='\s+', header=None, names=cols)
    test = pd.read_csv(f'{data_path}test_{data_id}.txt', sep='\s+', header=None, names=cols)
    rul = pd.read_csv(f'{data_path}RUL_{data_id}.txt', sep='\s+', header=None, names=['RUL'])
    return train, test, rul

def apply_gaussian(df, features, sigma):
    df = df.copy().sort_values(['unit_nr','time_cycles']).reset_index(drop=True)
    for uid in df['unit_nr'].unique():
        mask = df['unit_nr'] == uid
        for col in features:
            arr = df.loc[mask, col].to_numpy(dtype=np.float32)
            df.loc[mask, col] = gaussian_filter1d(arr, sigma=sigma, mode='nearest')
    return df

# ==========================================
# [1. 공통 전처리 셀]
# ==========================================
DATA_PATH = '../CMAPSSData/' # 실제 파일 경로에 맞게 수정하세요 ㅋ
CONFIG = {
    'DATA_ID': 'FD001',
    'RUL_CAP': 125,                # RUL Capping 값
    'GAUSS_SIGMA': 2.0,            # 가우시안 스무딩 시그마
    'SCALER_TYPE': 'minmax',       # 'minmax' 또는 'standard'
    'TEST_SIZE': 0.2,
    'RANDOM_STATE': 42,
    'BASE_SENSORS': ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 
                     's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21'],
    'SETTING_FEATURES': ['setting_1', 'setting_2', 'setting_3'],
    
    # ML 파생변수 세부 설정
    'ROLLING_WINDOW': 5,           
    'EMA_SPAN': 5,                 
    'LAG_SIZE': 3,                 
    'USE_MA': True, 
    'USE_DIFF': True, 
    'USE_LAG': True, 
    'USE_STD': True, 
    'USE_EMA': True
}

# 1. 데이터 로드 및 정렬
df_train, df_test, df_rul = load_cmapss_data(CONFIG['DATA_ID'], DATA_PATH)
df_train = df_train.sort_values(['unit_nr', 'time_cycles']).reset_index(drop=True)
df_test = df_test.sort_values(['unit_nr', 'time_cycles']).reset_index(drop=True)

# 2. RUL 생성 및 Capping (125)
df_train['max_cycle'] = df_train.groupby('unit_nr')['time_cycles'].transform('max')
df_train['RUL'] = (df_train['max_cycle'] - df_train['time_cycles']).clip(upper=CONFIG['RUL_CAP'])
df_train.drop(columns=['max_cycle'], inplace=True)

# 3. 가우시안 스무딩
df_train = apply_gaussian(df_train, CONFIG['BASE_SENSORS'], CONFIG['GAUSS_SIGMA'])
df_test = apply_gaussian(df_test, CONFIG['BASE_SENSORS'], CONFIG['GAUSS_SIGMA'])

# 4. 데이터 분리 (엔진 단위)
unit_ids = df_train['unit_nr'].unique()
train_units, val_units = train_test_split(unit_ids, test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_STATE'])

train_set = df_train[df_train['unit_nr'].isin(train_units)].copy()
val_set = df_train[df_train['unit_nr'].isin(val_units)].copy()
test_set = df_test.copy()

# 5. 정규화 (학습셋 기준)
X_features_common = CONFIG['BASE_SENSORS'] + CONFIG['SETTING_FEATURES']
scaler = MinMaxScaler() if CONFIG['SCALER_TYPE'] == 'minmax' else StandardScaler()

train_set[X_features_common] = scaler.fit_transform(train_set[X_features_common])
val_set[X_features_common] = scaler.transform(val_set[X_features_common])
test_set[X_features_common] = scaler.transform(test_set[X_features_common])

print("== 1. 공통 전처리 완료 (상수제거, RUL 125, 가우시안, 분할, 정규화) ==")
print("-" * 80)

In [ ]:
# ==========================================
# [2. ML 전처리 셀] - 파생 변수 생성
# ==========================================
def add_ml_features(df, features):
    df_res = df.copy().sort_values(['unit_nr', 'time_cycles'])
    for col in features:
        group = df_res.groupby('unit_nr')[col]
        if CONFIG['USE_MA']:
            df_res[f'{col}_ma'] = group.transform(lambda x: x.rolling(window=CONFIG['ROLLING_WINDOW'], min_periods=1).mean())
        if CONFIG['USE_STD']:
            df_res[f'{col}_std'] = group.transform(lambda x: x.rolling(window=CONFIG['ROLLING_WINDOW'], min_periods=1).std().fillna(0))
        if CONFIG['USE_DIFF']:
            df_res[f'{col}_diff'] = group.transform(lambda x: x.diff().fillna(0))
        if CONFIG['USE_EMA']:
            df_res[f'{col}_ema'] = group.transform(lambda x: x.ewm(span=CONFIG['EMA_SPAN']).mean())
        if CONFIG['USE_LAG']:
            for l in range(1, CONFIG['LAG_SIZE'] + 1):
                df_res[f'{col}_lag{l}'] = group.shift(l).fillna(method='bfill')
    return df_res

# 피처 생성 적용
train_ml = add_ml_features(train_set, CONFIG['BASE_SENSORS'])
val_ml = add_ml_features(val_set, CONFIG['BASE_SENSORS'])
test_ml = add_ml_features(test_set, CONFIG['BASE_SENSORS'])

# 최종 피처 리스트 업데이트
X_features_full = [c for c in train_ml.columns if c not in ['unit_nr', 'time_cycles', 'RUL']]

# 최종 변수 할당
X_train, y_train = train_ml[X_features_full], train_ml['RUL']
X_val, y_val = val_ml[X_features_full], val_ml['RUL']
X_test = test_ml.groupby('unit_nr').last()[X_features_full]
y_test = df_rul['RUL'].values

print("== 2. ML 파생변수 생성 완료 (MA, Std, Diff, EMA, Lag) ==")
print(f" 실험 설정 요약: {CONFIG['SCALER_TYPE']} / Sigma:{CONFIG['GAUSS_SIGMA']} / Window:{CONFIG['ROLLING_WINDOW']} / EMA_Span:{CONFIG['EMA_SPAN']}")
print(f" 최종 피처 수: {len(X_features_full)}개")
print(f" X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print(f" y_train: {y_train.shape}, y_val: {y_val.shape}, y_test: {y_test.shape}")
print("-" * 80)

In [ ]:
# ==========================================
# [3. DL 전처리 셀] - 슬라이딩 윈도우 생성
# ==========================================
SEQUENCE_LENGTH = 30 

def create_sequences(df, seq_length, features):
    data, labels = [], []
    for uid in df['unit_nr'].unique():
        udf = df[df['unit_nr'] == uid]
        if len(udf) < seq_length: continue
        feat_arr = udf[features].values
        label_arr = udf['RUL'].values
        for i in range(len(udf) - seq_length + 1):
            data.append(feat_arr[i:i+seq_length])
            labels.append(label_arr[i+seq_length-1])
    return np.array(data), np.array(labels)

# 윈도우 생성 (공통 전처리된 기본 피처 사용)
X_train_dl, y_train_dl = create_sequences(train_set, SEQUENCE_LENGTH, X_features_common)
X_val_dl, y_val_dl = create_sequences(val_set, SEQUENCE_LENGTH, X_features_common)

# 테스트 데이터 (마지막 윈도우 추출)
X_test_dl = []
for uid in test_set['unit_nr'].unique():
    udf = test_set[test_set['unit_nr'] == uid]
    last_seq = udf[X_features_common].values[-SEQUENCE_LENGTH:]
    if len(last_seq) < SEQUENCE_LENGTH:
        last_seq = np.pad(last_seq, ((SEQUENCE_LENGTH-len(last_seq), 0), (0, 0)), mode='constant')
    X_test_dl.append(last_seq)
X_test_dl = np.array(X_test_dl)

print("== 3. DL 슬라이딩 윈도우 생성 완료 (3D Tensor) ==")
print(f" 실험 설정 요약: {CONFIG['SCALER_TYPE']} / Sigma:{CONFIG['GAUSS_SIGMA']} / Seq_Length:{SEQUENCE_LENGTH}")
print(f" 최종 피처 수: {len(X_features_common)}개")
print(f" X_train: {X_train_dl.shape}, X_val: {X_val_dl.shape}, X_test: {X_test_dl.shape}")
print(f" y_train: {y_train_dl.shape}, y_val: {y_val_dl.shape}, y_test: {y_test.shape}")